# 🏥 HealthAI — Build Android APK with Buildozer

**Instructions:**
1. Upload your project as a ZIP (see Cell 1 – Upload)
2. Run each cell in order from top to bottom
3. At the end, download the generated `.apk` file

> ⏱ The full build takes **15–30 minutes** the first time (downloading Android SDK/NDK).
> Subsequent builds are faster because the cache is reused.

---

## Cell 0 — Check GPU/CPU Runtime
Make sure you're using a standard (CPU) runtime. Go to **Runtime → Change runtime type → CPU**.

In [ ]:
!uname -a
!python3 --version

## Cell 1 — Upload your project ZIP
Compress the entire `m--01` folder as `healthai.zip` (right-click → Send to → Compressed folder), then run this cell and upload it.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload healthai.zip here

## Cell 2 — Extract the project

In [ ]:
import zipfile, os

# Find the uploaded zip
zip_name = [f for f in os.listdir('.') if f.endswith('.zip')][0]
print(f'Extracting: {zip_name}')

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/healthai')

# Find the actual project root (handles nested folder in zip)
extracted = os.listdir('/content/healthai')
print('Extracted contents:', extracted)

# If there's a single subfolder, use it as root
if len(extracted) == 1 and os.path.isdir(f'/content/healthai/{extracted[0]}'):
    PROJECT_DIR = f'/content/healthai/{extracted[0]}'
else:
    PROJECT_DIR = '/content/healthai'

print(f'Project root: {PROJECT_DIR}')
os.listdir(PROJECT_DIR)

## Cell 3 — Install system dependencies

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq \
    python3-pip \
    build-essential \
    git \
    zip \
    unzip \
    openjdk-17-jdk \
    autoconf \
    libtool \
    pkg-config \
    zlib1g-dev \
    libncurses5-dev \
    libncursesw5-dev \
    libtinfo5 \
    cmake \
    libffi-dev \
    libssl-dev \
    libbz2-dev \
    libsqlite3-dev \
    ccache \
    lld

echo 'Java version:'
java -version

## Cell 4 — Install Buildozer and python-for-android

In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q buildozer
pip install -q cython==0.29.33
pip install -q python-for-android

buildozer --version
echo 'Buildozer installed successfully!'

## Cell 5 — Verify project files

In [ ]:
import os

# This should have been set in Cell 2
# If you need to re-run from here, set it manually:
# PROJECT_DIR = '/content/healthai'

required = [
    'main.py',
    'buildozer.spec',
    'screens/onboarding.py',
    'screens/home.py',
    'screens/disease_list.py',
    'screens/symptom_form.py',
    'screens/results.py',
    'models/predictor.py',
    'models/diabetes_model.pkl',
    'models/heart_model.pkl',
    'utils/theme.py',
    'data/form_schemas.py',
]

all_ok = True
for f in required:
    path = os.path.join(PROJECT_DIR, f)
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ MISSING'
    print(f'{status}  {f}')
    if not exists:
        all_ok = False

print()
print('All files present!' if all_ok else '⚠️  Some files are missing — check the ZIP.')

## Cell 6 — Fix buildozer.spec for Colab
We update the spec to use scikit-learn's Colab-compatible version and set correct paths.

In [ ]:
import os

spec_path = os.path.join(PROJECT_DIR, 'buildozer.spec')

with open(spec_path, 'r') as f:
    spec = f.read()

# Ensure correct requirements line (kivy 2.2.1 is well-tested with p4a)
spec = spec.replace(
    'requirements = python3,kivy==2.2.1,numpy,scikit-learn,joblib',
    'requirements = python3,kivy==2.2.1,numpy,scikit-learn,joblib'
)

# Remove the 'android.branch = master' line (use default stable)
lines = spec.splitlines()
lines = [l for l in lines if not l.strip().startswith('android.branch')]
spec = '\n'.join(lines)

with open(spec_path, 'w') as f:
    f.write(spec)

print('buildozer.spec updated.')
print('--- Current spec ---')
print(spec)

## Cell 7 — 🚀 Build the APK

This is the main build step. It will:
1. Download Android SDK (~1 GB)
2. Download Android NDK (~1 GB)
3. Compile all Python dependencies for ARM
4. Package everything into a `.apk`

> ⏱ **Expected time: 20–35 minutes** on first run.

In [ ]:
import os
os.chdir(PROJECT_DIR)
print(f'Building from: {os.getcwd()}')
!buildozer -v android debug 2>&1 | tail -200

## Cell 8 — Check build result

In [ ]:
import glob, os

apk_files = glob.glob(os.path.join(PROJECT_DIR, 'bin', '*.apk'))

if apk_files:
    for apk in apk_files:
        size_mb = os.path.getsize(apk) / (1024 * 1024)
        print(f'✅ APK built successfully!')
        print(f'   File: {apk}')
        print(f'   Size: {size_mb:.1f} MB')
else:
    print('❌ APK not found. Check the build log above for errors.')
    # Show last buildozer log
    log_files = glob.glob(os.path.join(PROJECT_DIR, '.buildozer', 'logs', '*.log'))
    if log_files:
        print(f'Build log: {log_files[-1]}')

## Cell 9 — ⬇️ Download the APK

In [ ]:
from google.colab import files
import glob, os

apk_files = glob.glob(os.path.join(PROJECT_DIR, 'bin', '*.apk'))

if apk_files:
    apk_path = apk_files[-1]  # Download the latest APK
    print(f'Downloading: {os.path.basename(apk_path)}')
    files.download(apk_path)
else:
    print('No APK found to download.')

---
## 🎉 Done!

Transfer the downloaded `.apk` to your Android phone:
- **Option A**: Email it to yourself and open on Android
- **Option B**: Use USB cable + file explorer
- **Option C**: Upload to Google Drive → open on phone

On the phone, go to **Settings → Install unknown apps** and allow installation from your browser/file manager.

> 🔁 For future builds (after code changes), just re-upload the ZIP and run Cells 1–2 and 7–9. The SDK/NDK are cached and it will be much faster (~5 min).